# nanochat base train + eval notebook (recreated)

Notebook-first workflow for `scripts/base_train.py` with research branch support.


## 1) Configure base training


In [ ]:
from math import ceil

TRAINING_CONFIG = {
    # Logging/runtime
    'run': 'nb-base-train',
    'device_type': '',
    'model_tag': 'nb-base-train',

    # Model/training
    'depth': 8,
    'aspect_ratio': 64,
    'head_dim': 64,
    'max_seq_len': 1024,
    'window_pattern': 'L',

    # Horizon
    'num_iterations': 200,
    'target_flops': -1.0,
    'target_param_data_ratio': -1.0,

    # Optimization
    'device_batch_size': 8,
    'total_batch_size': 2**16,
    'embedding_lr': 0.3,
    'unembedding_lr': 0.004,
    'weight_decay': 0.2,
    'matrix_lr': 0.02,
    'scalar_lr': 0.5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.95,
    'warmup_ratio': 0.1,
    'warmdown_ratio': 0.5,
    'final_lr_frac': 0.1,

    # Eval/checkpoint
    'resume_from_step': -1,
    'eval_every': 50,
    'eval_tokens': 8 * 524288,
    'core_metric_every': -1,
    'core_metric_max_per_task': 500,
    'sample_every': -1,
    'save_every': -1,
}


## 2) Convert config to argv for base_train


In [ ]:
def base_train_argv(cfg: dict) -> list[str]:
    flag_map = {
        'run': '--run', 'device_type': '--device-type',
        'depth': '--depth', 'aspect_ratio': '--aspect-ratio', 'head_dim': '--head-dim',
        'max_seq_len': '--max-seq-len', 'window_pattern': '--window-pattern',
        'num_iterations': '--num-iterations', 'target_flops': '--target-flops',
        'target_param_data_ratio': '--target-param-data-ratio',
        'device_batch_size': '--device-batch-size', 'total_batch_size': '--total-batch-size',
        'embedding_lr': '--embedding-lr', 'unembedding_lr': '--unembedding-lr',
        'weight_decay': '--weight-decay', 'matrix_lr': '--matrix-lr', 'scalar_lr': '--scalar-lr',
        'adam_beta1': '--adam-beta1', 'adam_beta2': '--adam-beta2',
        'warmup_ratio': '--warmup-ratio', 'warmdown_ratio': '--warmdown-ratio', 'final_lr_frac': '--final-lr-frac',
        'resume_from_step': '--resume-from-step',
        'eval_every': '--eval-every', 'eval_tokens': '--eval-tokens',
        'core_metric_every': '--core-metric-every', 'core_metric_max_per_task': '--core-metric-max-per-task',
        'sample_every': '--sample-every', 'save_every': '--save-every',
        'model_tag': '--model-tag',
    }
    argv = ['scripts.base_train']
    for k, flag in flag_map.items():
        if k in cfg and cfg[k] is not None:
            argv += [flag, str(cfg[k])]
    return argv

train_argv = base_train_argv(TRAINING_CONFIG)
print(' '.join(train_argv))


## 3) Run base_train (optional)


In [ ]:
# import runpy, sys
# old_argv = sys.argv[:]
# sys.argv = train_argv
# try:
#     runpy.run_module('scripts.base_train', run_name='__main__')
# finally:
#     sys.argv = old_argv


## 4) Research branch config


In [ ]:
RESEARCH_ALLOWED_KEYS = {
    "use_moe", "num_experts", "total_embed_dim", "router_dim", "capacity_factor",
    "use_sparse_top_k", "top_k", "routing_mode", "context_window",
    "causal", "use_expert_mlp", "use_output_projection",
    "use_expert_bias", "dropout", "use_shared_base", "shared_base_dim",
    "use_vocab_prior", "expert_residual", "allow_replacement",
    "use_embed_refine", "target_dim", "selection_mode", "use_perm",
    "use_remixed_linear", "context_dim", "linear_basis_size", "remixed_linear_kwargs",
    "moe_use_abs_pos_embed", "research_onecycle", "research_warmup_ratio",
    # dedicated research LR controls
    "research_embedding_lr", "research_unembedding_lr",
}

RESEARCH_MODEL_CONFIG = {
    "use_moe": True,
    "use_perm": True,
    "num_experts": 8,
    "router_dim": 64,
    "target_dim": 256,
    "selection_mode": "soft",
    "allow_replacement": True,
    "use_remixed_linear": True,
    "context_dim": 64,
    "linear_basis_size": 64,
    "moe_use_abs_pos_embed": 1,
    "research_onecycle": 1,
    "research_warmup_ratio": 0.1,
    # these override only for research runs
    "research_embedding_lr": 0.6,
    "research_unembedding_lr": 0.008,
    "remixed_linear_kwargs": {
        "use_basis_gate": True,
        "use_output_gate": True,
        "use_context": True,
    },
}

def research_to_gpt_kwargs(cfg: dict) -> dict:
    unexpected = sorted(set(cfg) - RESEARCH_ALLOWED_KEYS)
    if unexpected:
        print(f"Warning: unexpected research keys: {unexpected}")
    return {
        "use_moe": cfg.get("use_moe", False),
        "use_perm": cfg.get("use_perm", False),
        "num_experts": cfg.get("num_experts", 8),
        "router_dim": cfg.get("router_dim", 64),
        "target_dim": cfg.get("target_dim", 64),
        "selection_mode": cfg.get("selection_mode", "soft"),
        "allow_replacement": cfg.get("allow_replacement", True),
        "use_remixed_linear": cfg.get("use_remixed_linear", False),
        "context_dim": cfg.get("context_dim", 64),
        "linear_basis_size": cfg.get("linear_basis_size", 64),
        "moe_use_abs_pos_embed": cfg.get("moe_use_abs_pos_embed", 1),
        "research_onecycle": cfg.get("research_onecycle", 1),
        "research_warmup_ratio": cfg.get("research_warmup_ratio", 0.1),
        "research_embedding_lr": cfg.get("research_embedding_lr", None),
        "research_unembedding_lr": cfg.get("research_unembedding_lr", None),
        "remixed_linear_kwargs": cfg.get("remixed_linear_kwargs", {
            "use_basis_gate": True,
            "use_output_gate": True,
            "use_context": True,
        }),
    }

RESEARCH_GPT_KWARGS = research_to_gpt_kwargs(RESEARCH_MODEL_CONFIG)
RESEARCH_GPT_KWARGS


## 5) Build research argv (includes dedicated research embedding/unembedding LR)


In [ ]:
MOE_SCALE = 5.0  # research LR multiplier vs base LR defaults

def research_train_argv(training_cfg: dict, research_kwargs: dict, moe_scale: float = MOE_SCALE) -> list[str]:
    argv = base_train_argv(training_cfg)

    # bool flags
    if research_kwargs.get("use_moe", False):
        argv += ["--use-moe"]
    if research_kwargs.get("use_perm", False):
        argv += ["--use-perm"]
    if research_kwargs.get("allow_replacement", False):
        argv += ["--allow-replacement"]
    if research_kwargs.get("use_remixed_linear", False):
        argv += ["--use-remixed-linear"]

    # scalar flags
    scalar_map = {
        "num_experts": "--num-experts",
        "router_dim": "--router-dim",
        "target_dim": "--target-dim",
        "selection_mode": "--selection-mode",
        "context_dim": "--context-dim",
        "linear_basis_size": "--linear-basis-size",
        "moe_use_abs_pos_embed": "--moe-use-abs-pos-embed",
        "research_onecycle": "--research-onecycle",
        "research_warmup_ratio": "--research-warmup-ratio",
    }
    for k, flag in scalar_map.items():
        if k in research_kwargs and research_kwargs[k] is not None:
            argv += [flag, str(research_kwargs[k])]

    # remixed linear gate controls
    remix_kwargs = research_kwargs.get("remixed_linear_kwargs", {})
    argv += ["--remix-use-basis-gate", str(int(bool(remix_kwargs.get("use_basis_gate", True))))]
    argv += ["--remix-use-output-gate", str(int(bool(remix_kwargs.get("use_output_gate", True))))]
    argv += ["--remix-use-context", str(int(bool(remix_kwargs.get("use_context", True))))]

    # dedicated research LR overrides (if provided), else MOE_SCALE fallback
    emb = research_kwargs.get("research_embedding_lr", None)
    unemb = research_kwargs.get("research_unembedding_lr", None)
    if emb is None:
        emb = float(training_cfg["embedding_lr"]) * float(moe_scale)
    if unemb is None:
        unemb = float(training_cfg["unembedding_lr"]) * float(moe_scale)
    argv += ["--embedding-lr", str(emb)]
    argv += ["--unembedding-lr", str(unemb)]

    # keep existing scaling behavior for matrix/scalar
    argv += ["--matrix-lr", str(float(training_cfg["matrix_lr"]) * float(moe_scale))]
    argv += ["--scalar-lr", str(float(training_cfg["scalar_lr"]) * float(moe_scale))]

    return argv

# Ensure target_dim % head_dim == 0 in research runs
TRAINING_CONFIG['head_dim'] = 64

research_argv = research_train_argv(TRAINING_CONFIG, RESEARCH_GPT_KWARGS, moe_scale=MOE_SCALE)
print(' '.join(research_argv))


## 6) Run research train (optional)


In [ ]:
# import runpy, sys
# old_argv = sys.argv[:]
# sys.argv = research_argv
# try:
#     runpy.run_module('scripts.base_train', run_name='__main__')
# finally:
#     sys.argv = old_argv
